# YOLOX MSL Fine-Tuning: Add Military Shipping Label Detection

Fine-tune the 97-class confidence-boost model to add military shipping label (MSL) as class 98.

## Strategy
- **Start from**: `confidence_boost_epoch_60.pth` (100 base + 60 confidence boost epochs)
- **Expand head**: 97 → 98 classes (add 1 neuron for MSL)
- **Full dataset replay**: All 2262 original images + MSL images (5x oversampled)
- **Two-phase training**: 15 epochs frozen backbone + 25 epochs full fine-tune
- **Total**: 40 epochs, ~2-3 hours on T4

## Prerequisites
1. Upload MSL images + annotations to Google Drive (see Pre-Flight below)
2. Runtime set to **GPU T4** or better

## Pre-Flight: Upload to Google Drive
1. `assets/MSLImages/` → `/My Drive/HazProML/data/MSLImages/`
2. `assets/MSLImages_annotations/` → `/My Drive/HazProML/data/MSLImages_annotations/`
3. Verify `/My Drive/HazProML/data/combined_dataset/` exists
4. Verify `/My Drive/HazProML/models/confidence_boost_97class/confidence_boost_epoch_60.pth` exists

## Cell 1: Mount Google Drive & Verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU! Enable in Runtime > Change runtime type")

## Cell 2: Configuration

In [ ]:
import os
import json

# ============================================
# PATHS
# ============================================
DRIVE_ROOT = "/content/drive/MyDrive"

# Original combined dataset (97 classes, 2262 images)
COMBINED_DATASET = f"{DRIVE_ROOT}/HazProML/data/combined_dataset"

# MSL dataset 1: primary set (51 MSL bboxes)
MSL_IMAGES_DIR = f"{DRIVE_ROOT}/HazProML/data/MSLImages"
MSL_ANNOTATIONS_DIR = f"{DRIVE_ROOT}/HazProML/data/MSLImages_annotations"

# MSL dataset 2: additional images (22 MSL bboxes, partially overlaps dataset 1)
MSL2_IMAGES_DIR = f"{DRIVE_ROOT}/HazProML/data/real_hazmat_images_WITH_MSL"
MSL2_ANNOTATIONS_DIR = f"{DRIVE_ROOT}/HazProML/data/real_hazmat_images_WITH_MSL_annotations"

# Confidence boost checkpoint (best model)
PRETRAINED_CKPT = f"{DRIVE_ROOT}/HazProML/models/confidence_boost_97class/confidence_boost_epoch_60.pth"

# Output
DRIVE_OUTPUT = f"{DRIVE_ROOT}/HazProML/models/msl_finetune_98class"

# Local working directory
LOCAL_DATA_DIR = '/content/data/msl_finetune'

# ============================================
# MODEL PARAMETERS
# ============================================
OLD_NUM_CLASSES = 97   # Current model
NEW_NUM_CLASSES = 98   # After adding MSL
MSL_CLASS_ID = 97      # 0-indexed class ID for MSL
INPUT_SIZE = (640, 640)
BATCH_SIZE = 16

# ============================================
# TRAINING PARAMETERS
# ============================================
# Phase 1: Frozen backbone (head warmup)
PHASE1_EPOCHS = 15
PHASE1_LR = 0.001 / 64.0  # 2x higher than confidence boost (new neuron needs to learn)

# Phase 2: Full fine-tune (separate run, loads Phase 1 weights via -c)
PHASE2_EPOCHS = 60         # Phase 2 runs its own 60 epochs (starts from 0)
PHASE2_LR = 0.0005 / 64.0 # Same as confidence boost
NO_AUG_EPOCHS = 15         # Last 15 epochs without augmentation

# Confidence boost settings (carried over)
OBJ_LOSS_WEIGHT = 2.0
SAVE_INTERVAL = 5

# MSL oversampling factor
MSL_OVERSAMPLE = 10

# ============================================
# VERIFY PATHS
# ============================================
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print("=" * 60)
print("MSL FINE-TUNING CONFIGURATION")
print("=" * 60)

paths = {
    "Combined dataset": COMBINED_DATASET,
    "MSL images (set 1)": MSL_IMAGES_DIR,
    "MSL annotations (set 1)": MSL_ANNOTATIONS_DIR,
    "MSL images (set 2)": MSL2_IMAGES_DIR,
    "MSL annotations (set 2)": MSL2_ANNOTATIONS_DIR,
    "Pretrained checkpoint": PRETRAINED_CKPT,
}

all_found = True
for name, path in paths.items():
    exists = os.path.exists(path)
    if not exists:
        all_found = False
    size_info = ""
    if exists and os.path.isfile(path):
        size_info = f" ({os.path.getsize(path)/1024/1024:.1f} MB)"
    print(f"  {name}: {'FOUND' if exists else 'NOT FOUND'}{size_info}")
    print(f"    {path}")

if not all_found:
    print("\n*** MISSING FILES! Upload all datasets to Google Drive. ***")
else:
    print(f"\nAll paths verified!")

print(f"\nTraining plan:")
print(f"  Phase 1: {PHASE1_EPOCHS} epochs (frozen backbone, LR={PHASE1_LR*64:.4f})")
print(f"  Phase 2: {PHASE2_EPOCHS} epochs (full fine-tune, LR={PHASE2_LR*64:.6f})")
print(f"  No-aug: last {NO_AUG_EPOCHS} epochs of Phase 2")
print(f"  OBJ_LOSS_WEIGHT: {OBJ_LOSS_WEIGHT}")
print(f"  MSL oversample: {MSL_OVERSAMPLE}x")
print(f"\nOutput: {DRIVE_OUTPUT}")
print("=" * 60)

## Cell 3: Install YOLOX

In [ ]:
%%capture
!pip install cython pycocotools thop loguru tabulate

import os
if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

%cd /content/YOLOX
!pip install -v -e .

In [ ]:
import sys
sys.path.insert(0, '/content/YOLOX')
from yolox.exp import get_exp
print("YOLOX installed successfully!")

## Cell 4: Prepare Merged Dataset

Combines the original 2262-image dataset with MSL images (5x oversampled).

MSL images are split 80/20 into train/val so we can validate MSL detection.

In [ ]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

random.seed(42)

# Clean and create directories
!rm -rf /content/data
for split in ['train', 'val']:
    os.makedirs(f'{LOCAL_DATA_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{LOCAL_DATA_DIR}/labels/{split}', exist_ok=True)

# ============================================
# STEP 1: Copy original combined dataset
# ============================================
print("Step 1: Copying original combined dataset (2262 images)...")
print(f"  Source: {COMBINED_DATASET}")

orig_train_imgs = f"{COMBINED_DATASET}/images/train"
orig_train_labels = f"{COMBINED_DATASET}/labels/train"
orig_val_imgs = f"{COMBINED_DATASET}/images/val"
orig_val_labels = f"{COMBINED_DATASET}/labels/val"

orig_train_count = 0
if os.path.exists(orig_train_imgs):
    for f in tqdm(os.listdir(orig_train_imgs), desc="  Train images"):
        src_img = f"{orig_train_imgs}/{f}"
        if not os.path.isfile(src_img):
            continue
        shutil.copy2(src_img, f"{LOCAL_DATA_DIR}/images/train/{f}")
        # Copy matching label
        stem = os.path.splitext(f)[0]
        src_label = f"{orig_train_labels}/{stem}.txt"
        if os.path.exists(src_label):
            shutil.copy2(src_label, f"{LOCAL_DATA_DIR}/labels/train/{stem}.txt")
        orig_train_count += 1

orig_val_count = 0
if os.path.exists(orig_val_imgs):
    for f in os.listdir(orig_val_imgs):
        src_img = f"{orig_val_imgs}/{f}"
        if not os.path.isfile(src_img):
            continue
        shutil.copy2(src_img, f"{LOCAL_DATA_DIR}/images/val/{f}")
        stem = os.path.splitext(f)[0]
        src_label = f"{orig_val_labels}/{stem}.txt"
        if os.path.exists(src_label):
            shutil.copy2(src_label, f"{LOCAL_DATA_DIR}/labels/val/{stem}.txt")
        orig_val_count += 1

print(f"  Copied {orig_train_count} train + {orig_val_count} val images")

# ============================================
# STEP 2: Collect MSL images from BOTH datasets
# ============================================
print(f"\nStep 2: Collecting MSL images from both datasets...")

def collect_msl_pairs(images_dir, annotations_dir, dataset_name):
    """Collect (image_path, label_path, basename) from one MSL dataset."""
    ann_data_dir = f"{annotations_dir}/obj_train_data"
    train_txt = f"{annotations_dir}/train.txt"
    pairs = []

    if not os.path.exists(train_txt):
        print(f"  WARNING: {train_txt} not found, skipping {dataset_name}")
        return pairs

    with open(train_txt, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            img_basename = os.path.basename(line)
            img_stem = os.path.splitext(img_basename)[0]
            img_path = f"{images_dir}/{img_basename}"
            label_path = f"{ann_data_dir}/{img_stem}.txt"

            if os.path.exists(img_path) and os.path.exists(label_path):
                pairs.append((img_path, label_path, img_basename))
            elif not os.path.exists(img_path):
                print(f"  WARNING: Image not found in {dataset_name}: {img_path}")

    print(f"  {dataset_name}: {len(pairs)} image+annotation pairs")
    return pairs

# Collect from dataset 1 (primary - 51 MSL bboxes)
msl1_pairs = collect_msl_pairs(MSL_IMAGES_DIR, MSL_ANNOTATIONS_DIR, "MSL dataset 1")

# Collect from dataset 2 (additional - 22 MSL bboxes)
msl2_pairs = collect_msl_pairs(MSL2_IMAGES_DIR, MSL2_ANNOTATIONS_DIR, "MSL dataset 2")

# Merge with deduplication by filename (dataset 1 takes priority)
seen_basenames = set(basename for _, _, basename in msl1_pairs)
msl_pairs = list(msl1_pairs)
added_from_msl2 = 0
for img_path, label_path, basename in msl2_pairs:
    if basename not in seen_basenames:
        msl_pairs.append((img_path, label_path, basename))
        seen_basenames.add(basename)
        added_from_msl2 += 1
    else:
        pass  # Skip duplicate

print(f"\n  Total after dedup: {len(msl_pairs)} unique MSL pairs")
print(f"    From dataset 1: {len(msl1_pairs)}")
print(f"    Added from dataset 2: {added_from_msl2}")
print(f"    Duplicates skipped: {len(msl2_pairs) - added_from_msl2}")

# Count MSL bounding boxes
msl_bbox_count = 0
for _, label_path, _ in msl_pairs:
    with open(label_path, 'r') as f:
        for line in f:
            if line.strip().startswith(f"{MSL_CLASS_ID} "):
                msl_bbox_count += 1
print(f"  Total MSL bounding boxes: {msl_bbox_count}")

# ============================================
# STEP 3: Split MSL into train/val (80/20)
# ============================================
print(f"\nStep 3: Splitting MSL images 80/20...")

random.shuffle(msl_pairs)
val_size = max(1, len(msl_pairs) // 5)  # 20% for validation
msl_val = msl_pairs[:val_size]
msl_train = msl_pairs[val_size:]

print(f"  MSL train: {len(msl_train)} images")
print(f"  MSL val: {len(msl_val)} images")

# ============================================
# STEP 4: Copy MSL images with oversampling
# ============================================
print(f"\nStep 4: Copying MSL images ({MSL_OVERSAMPLE}x oversampled)...")

msl_train_copied = 0
for img_path, label_path, img_basename in msl_train:
    img_stem = os.path.splitext(img_basename)[0]
    img_ext = os.path.splitext(img_basename)[1]

    for dup in range(MSL_OVERSAMPLE):
        prefix = f"msl_d{dup}_"
        dst_img = f"{LOCAL_DATA_DIR}/images/train/{prefix}{img_basename}"
        dst_label = f"{LOCAL_DATA_DIR}/labels/train/{prefix}{img_stem}.txt"
        shutil.copy2(img_path, dst_img)
        shutil.copy2(label_path, dst_label)
        msl_train_copied += 1

# Copy MSL validation images (no oversampling)
msl_val_copied = 0
for img_path, label_path, img_basename in msl_val:
    img_stem = os.path.splitext(img_basename)[0]
    dst_img = f"{LOCAL_DATA_DIR}/images/val/msl_{img_basename}"
    dst_label = f"{LOCAL_DATA_DIR}/labels/val/msl_{img_stem}.txt"
    shutil.copy2(img_path, dst_img)
    shutil.copy2(label_path, dst_label)
    msl_val_copied += 1

print(f"  MSL train copies: {msl_train_copied} ({len(msl_train)} images x {MSL_OVERSAMPLE})")
print(f"  MSL val copies: {msl_val_copied}")

# ============================================
# STEP 5: Create 98-class mapping
# ============================================
print(f"\nStep 5: Creating 98-class mapping...")

# Load original class mapping
orig_mapping_path = f"{COMBINED_DATASET}/class_mapping.json"
if os.path.exists(orig_mapping_path):
    with open(orig_mapping_path, 'r') as f:
        class_map = json.load(f)
else:
    raise FileNotFoundError(f"Class mapping not found: {orig_mapping_path}")

# Handle both dict and list formats
if isinstance(class_map, list):
    new_class_map = list(class_map)
    new_class_map.append({
        "id": MSL_CLASS_ID,
        "name": "militaryShippingLabel",
        "category": "military"
    })
elif isinstance(class_map, dict):
    new_class_map = dict(class_map)
    new_class_map[str(MSL_CLASS_ID)] = {
        "name": "militaryShippingLabel",
        "category": "military"
    }

with open(f"{LOCAL_DATA_DIR}/class_mapping.json", 'w') as f:
    json.dump(new_class_map, f, indent=2)
print(f"  Saved 98-class mapping to {LOCAL_DATA_DIR}/class_mapping.json")

# ============================================
# SUMMARY
# ============================================
total_train = len(os.listdir(f"{LOCAL_DATA_DIR}/images/train"))
total_val = len(os.listdir(f"{LOCAL_DATA_DIR}/images/val"))

print(f"\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Training images: {total_train}")
print(f"  - Original: {orig_train_count}")
print(f"  - MSL (oversampled): {msl_train_copied}")
print(f"  - MSL effective ratio: {msl_train_copied/total_train*100:.1f}%")
print(f"Validation images: {total_val}")
print(f"  - Original: {orig_val_count}")
print(f"  - MSL: {msl_val_copied}")
print(f"MSL sources: 2 datasets ({len(msl1_pairs)} + {added_from_msl2} unique)")
print(f"MSL bounding boxes: {msl_bbox_count}")
print(f"Classes: {NEW_NUM_CLASSES}")
print("=" * 60)

## Cell 5: Convert YOLO → COCO Format

In [ ]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime

def yolo_to_coco(images_dir, labels_dir, class_mapping_path, num_classes, output_path):
    """Convert YOLO annotations to COCO format for 98 classes."""

    # Build categories list
    with open(class_mapping_path, 'r') as f:
        class_map = json.load(f)

    if isinstance(class_map, list):
        categories = [{"id": c["id"], "name": c["name"], "supercategory": c.get("category", "hazmat")}
                      for c in class_map]
    elif isinstance(class_map, dict):
        categories = [{"id": int(k), "name": v["name"], "supercategory": v.get("category", "hazmat")}
                      for k, v in class_map.items()]

    images, annotations = [], []
    annotation_id = 0

    # Find all image files
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
        image_files.extend(Path(images_dir).glob(ext))
    image_files = sorted(image_files)

    for img_id, img_path in enumerate(tqdm(image_files, desc="Converting")):
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"Error reading {img_path}: {e}")
            continue

        images.append({
            "id": img_id, "file_name": img_path.name,
            "width": width, "height": height, "license": 1,
            "date_captured": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        with open(label_path, 'r') as f:
            content = f.read().strip()
        if not content:
            continue

        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            if class_id >= num_classes:
                print(f"  WARNING: class_id {class_id} >= {num_classes} in {label_path.name}, skipping")
                continue
            x_center, y_center, box_w, box_h = map(float, parts[1:])

            x = (x_center - box_w / 2) * width
            y = (y_center - box_h / 2) * height
            w = box_w * width
            h = box_h * height

            annotations.append({
                "id": annotation_id, "image_id": img_id, "category_id": class_id,
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2), "iscrowd": 0, "segmentation": []
            })
            annotation_id += 1

    coco = {
        "info": {"description": "HazProML MSL Fine-Tuning Dataset", "version": "1.0", "year": 2025},
        "licenses": [{"id": 1, "name": "MIT", "url": ""}],
        "categories": categories, "images": images, "annotations": annotations
    }
    with open(output_path, 'w') as f:
        json.dump(coco, f)
    return len(images), len(annotations)

# Create annotations directory
os.makedirs(f'{LOCAL_DATA_DIR}/annotations', exist_ok=True)

print("Converting training set...")
train_imgs, train_anns = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/train', f'{LOCAL_DATA_DIR}/labels/train',
    f'{LOCAL_DATA_DIR}/class_mapping.json', NEW_NUM_CLASSES,
    f'{LOCAL_DATA_DIR}/annotations/train.json'
)
print(f"  {train_imgs} images, {train_anns} annotations")

print("\nConverting validation set...")
val_imgs, val_anns = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/val', f'{LOCAL_DATA_DIR}/labels/val',
    f'{LOCAL_DATA_DIR}/class_mapping.json', NEW_NUM_CLASSES,
    f'{LOCAL_DATA_DIR}/annotations/val.json'
)
print(f"  {val_imgs} images, {val_anns} annotations")

# Count MSL annotations in COCO format
with open(f'{LOCAL_DATA_DIR}/annotations/train.json', 'r') as f:
    train_coco = json.load(f)
msl_train_anns = sum(1 for a in train_coco['annotations'] if a['category_id'] == MSL_CLASS_ID)
print(f"\nMSL annotations in training set: {msl_train_anns}")
print("COCO conversion complete!")

## Cell 6: Create YOLOX Config + Patch OBJ_LOSS_WEIGHT

Creates two experiment configs:
- **Phase 1**: Frozen backbone, higher LR (head warmup, 15 epochs)
- **Phase 2**: All layers trainable, lower LR (full fine-tune, 25 more epochs)

Also patches `yolo_head.py` to apply 2x objectness loss weight.

In [ ]:
import re

# ============================================
# PHASE 1 CONFIG: Frozen backbone
# ============================================
phase1_config = f'''#!/usr/bin/env python3
import os
import torch
import torch.nn as nn
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.375
        self.num_classes = {NEW_NUM_CLASSES}
        self.act = "silu"

        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.data_num_workers = 2

        self.max_epoch = {PHASE1_EPOCHS}
        self.warmup_epochs = 3
        self.basic_lr_per_img = {PHASE1_LR}
        self.scheduler = "yoloxwarmcos"
        self.min_lr_ratio = 0.05
        self.no_aug_epochs = 0  # Keep augmentation ON for Phase 1

        self.input_size = {INPUT_SIZE}
        self.test_size = {INPUT_SIZE}
        self.random_size = (14, 26)

        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.enable_mixup = True
        self.flip_prob = 0.5
        self.hsv_prob = 1.0

        self.nmsthre = 0.65
        self.test_conf = 0.01

        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_msl_finetune_phase1"
        self.eval_interval = 99999
        self.print_interval = 50
        self.save_history_ckpt = True

    def get_model(self):
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        in_channels = [256, 512, 1024]
        backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act)
        head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act)
        self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)

        # FREEZE BACKBONE for Phase 1
        for param in self.model.backbone.parameters():
            param.requires_grad = False

        return self.model

    def get_optimizer(self, batch_size):
        """Only optimize head parameters (backbone is frozen)."""
        if "optimizer" not in self.__dict__:
            if self.warmup_epochs > 0:
                lr = self.warmup_lr
            else:
                lr = self.basic_lr_per_img * batch_size

            pg0, pg1, pg2 = [], [], []
            for k, v in self.model.head.named_modules():
                if hasattr(v, "bias") and isinstance(v.bias, nn.Parameter):
                    pg2.append(v.bias)
                if isinstance(v, nn.BatchNorm2d) or "bn" in k:
                    pg0.append(v.weight)
                elif hasattr(v, "weight") and isinstance(v.weight, nn.Parameter):
                    pg1.append(v.weight)

            optimizer = torch.optim.SGD(pg0, lr=lr, momentum=self.momentum, nesterov=True)
            optimizer.add_param_group({{"params": pg1, "weight_decay": self.weight_decay}})
            optimizer.add_param_group({{"params": pg2}})
            self.optimizer = optimizer
        return self.optimizer

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master
        with wait_for_the_master():
            dataset = COCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob), cache=False, name="images/train")
        dataset = MosaicDetection(dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            degrees=10.0, translate=0.1, mosaic_scale=(0.1, 2.0), mixup_scale=(0.5, 1.5),
            shear=2.0, enable_mixup=self.enable_mixup, mosaic_prob=self.mosaic_prob, mixup_prob=self.mixup_prob)
        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=self.data_num_workers, pin_memory=True,
                          batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)

    def get_eval_loader(self, *args, **kwargs): return None
    def get_evaluator(self, *args, **kwargs): return None
    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        return (0, 0, "Evaluation disabled"), None
'''

with open('/content/YOLOX/exps/msl_phase1_exp.py', 'w') as f:
    f.write(phase1_config)
print("Created Phase 1 config (frozen backbone)")

# ============================================
# PHASE 2 CONFIG: Full fine-tune
# ============================================
phase2_config = f'''#!/usr/bin/env python3
import os
import torch
import torch.nn as nn
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.375
        self.num_classes = {NEW_NUM_CLASSES}
        self.act = "silu"

        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.data_num_workers = 2

        self.max_epoch = {PHASE2_EPOCHS}
        self.warmup_epochs = 2
        self.basic_lr_per_img = {PHASE2_LR}
        self.scheduler = "yoloxwarmcos"
        self.min_lr_ratio = 0.01
        self.no_aug_epochs = {NO_AUG_EPOCHS}

        self.input_size = {INPUT_SIZE}
        self.test_size = {INPUT_SIZE}
        self.random_size = (14, 26)

        self.mosaic_prob = 0.5
        self.mixup_prob = 0.3
        self.enable_mixup = True
        self.flip_prob = 0.5
        self.hsv_prob = 1.0

        self.nmsthre = 0.65
        self.test_conf = 0.01

        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_msl_finetune_phase2"
        self.eval_interval = 99999
        self.print_interval = 50
        self.save_history_ckpt = True

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master
        with wait_for_the_master():
            dataset = COCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob), cache=False, name="images/train")
        dataset = MosaicDetection(dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            degrees=10.0, translate=0.1, mosaic_scale=(0.1, 2.0), mixup_scale=(0.5, 1.5),
            shear=2.0, enable_mixup=self.enable_mixup, mosaic_prob=self.mosaic_prob, mixup_prob=self.mixup_prob)
        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=self.data_num_workers, pin_memory=True,
                          batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)

    def get_eval_loader(self, *args, **kwargs): return None
    def get_evaluator(self, *args, **kwargs): return None
    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        return (0, 0, "Evaluation disabled"), None
'''

with open('/content/YOLOX/exps/msl_phase2_exp.py', 'w') as f:
    f.write(phase2_config)
print("Created Phase 2 config (full fine-tune)")

# ============================================
# PATCH yolo_head.py for OBJ_LOSS_WEIGHT
# ============================================
yolo_head_path = '/content/YOLOX/yolox/models/yolo_head.py'
with open(yolo_head_path, 'r') as f:
    content = f.read()

if f'OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}' in content:
    print(f"\nyolo_head.py already patched with OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}")
else:
    # Add constant before class definition
    insert_pos = content.find('class YOLOXHead')
    if insert_pos > 0:
        content = content[:insert_pos] + f'\nOBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}\n\n' + content[insert_pos:]

    # Multiply loss_obj by OBJ_LOSS_WEIGHT
    lines = content.split('\n')
    for i, line in enumerate(lines):
        if 'loss_obj' in line and 'bcewithlog_loss' in line and 'OBJ_LOSS_WEIGHT' not in line:
            lines[i] = line.rstrip() + ' * OBJ_LOSS_WEIGHT  # MSL FINETUNE'
            print(f"\nPatched line {i+1}: {lines[i].strip()[:80]}...")
            break

    content = '\n'.join(lines)
    with open(yolo_head_path, 'w') as f:
        f.write(content)
    print(f"Patched yolo_head.py with OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}")

## Cell 7: Expand Model Head (97 → 98 classes)

Loads the confidence boost checkpoint, creates a new 98-class model, and copies all existing weights.
The new MSL neuron (class 97) is initialized with Xavier weights + negative bias.

In [ ]:
import torch
import torch.nn as nn
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

def create_yolox(num_classes):
    """Create YOLOX-Tiny model."""
    in_channels = [256, 512, 1024]
    backbone = YOLOPAFPN(0.33, 0.375, in_channels=in_channels, act='silu')
    head = YOLOXHead(num_classes, 0.375, in_channels=in_channels, act='silu')
    return YOLOX(backbone, head)

# Load old checkpoint
print("Loading confidence boost checkpoint...")
ckpt = torch.load(PRETRAINED_CKPT, map_location='cpu')
old_state = ckpt['model']
print(f"  Source epoch: {ckpt.get('start_epoch', 'unknown')}")

# Create old model (97 classes) to verify weights load correctly
old_model = create_yolox(OLD_NUM_CLASSES)
old_model.load_state_dict(old_state)
print(f"  Old model loaded: {OLD_NUM_CLASSES} classes")

# Create new model (98 classes)
new_model = create_yolox(NEW_NUM_CLASSES)
new_state = new_model.state_dict()

# Copy weights, expanding cls_preds layers
expanded_keys = []
for key in new_state:
    if key in old_state:
        if 'cls_preds' in key:
            old_tensor = old_state[key]
            new_tensor = new_state[key]

            if 'weight' in key:
                # Conv2d weight: [out_channels, in_channels, 1, 1]
                new_tensor[:OLD_NUM_CLASSES] = old_tensor
                # Initialize MSL channel with Xavier
                nn.init.xavier_uniform_(new_tensor[OLD_NUM_CLASSES:NEW_NUM_CLASSES])
                expanded_keys.append(f"{key} [{old_tensor.shape[0]}→{new_tensor.shape[0]}]")
            elif 'bias' in key:
                # Bias: [out_channels]
                new_tensor[:OLD_NUM_CLASSES] = old_tensor
                # Small negative bias so MSL starts suppressed (avoids false positive flood)
                new_tensor[OLD_NUM_CLASSES:NEW_NUM_CLASSES] = -2.0
                expanded_keys.append(f"{key} [{old_tensor.shape[0]}→{new_tensor.shape[0]}]")

            new_state[key] = new_tensor
        else:
            # Direct copy for all non-cls_preds layers
            new_state[key] = old_state[key]

new_model.load_state_dict(new_state)
print(f"\n  New model created: {NEW_NUM_CLASSES} classes")
print(f"  Expanded layers:")
for k in expanded_keys:
    print(f"    {k}")

# Verify with dummy forward pass
dummy = torch.randn(1, 3, 640, 640)
new_model.eval()
with torch.no_grad():
    out = new_model(dummy)
print(f"\n  Output shape: {out.shape}")
expected_shape = (1, 8400, 5 + NEW_NUM_CLASSES)
assert out.shape == torch.Size(expected_shape), f"Expected {expected_shape}, got {out.shape}"
print(f"  Output shape verified: [batch, {8400} anchors, {5 + NEW_NUM_CLASSES} (5 + {NEW_NUM_CLASSES} classes)]")

# Save expanded checkpoint for Phase 1
EXPANDED_CKPT = '/content/outputs/yolox_msl_finetune/expanded_98class_ckpt.pth'
os.makedirs(os.path.dirname(EXPANDED_CKPT), exist_ok=True)
torch.save({
    'model': new_model.state_dict(),
    'start_epoch': 0,  # Reset epoch counter
}, EXPANDED_CKPT)
print(f"\n  Saved expanded checkpoint: {EXPANDED_CKPT}")
print(f"  Epoch counter reset to 0")
print(f"\nHead expansion complete!")

## Cell 8: Phase 1 — Head Warmup (15 epochs, frozen backbone)

Backbone is frozen. Only the detection head trains. The new MSL neuron learns to fire on MSL images while existing neurons are reinforced by the full dataset replay.

**Expected time: ~30-45 min on T4**

In [ ]:
import shutil

PHASE1_OUTPUT_DIR = '/content/outputs/yolox_msl_finetune_phase1'
os.makedirs(PHASE1_OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print(f"PHASE 1: HEAD WARMUP ({PHASE1_EPOCHS} EPOCHS)")
print(f"  Backbone: FROZEN")
print(f"  LR: {PHASE1_LR * 64:.4f}")
print(f"  Augmentation: ON (mosaic + mixup)")
print("=" * 60)

%cd /content/YOLOX

!PYTHONPATH=/content/YOLOX python tools/train.py \
    -f exps/msl_phase1_exp.py \
    -d 1 \
    -b {BATCH_SIZE} \
    --fp16 \
    -o \
    -c {EXPANDED_CKPT}

print("\n" + "=" * 60)
print("PHASE 1 COMPLETE!")
print("=" * 60)

# Save Phase 1 checkpoint to Drive
shutil.copy(f'{PHASE1_OUTPUT_DIR}/latest_ckpt.pth',
            f'{DRIVE_OUTPUT}/phase1_epoch_{PHASE1_EPOCHS}.pth')
print(f"Saved phase 1 checkpoint to Drive")

## Cell 9: Phase 2 — Full Fine-Tune (25 epochs, all layers trainable)

All layers are unfrozen. Lower LR + cosine decay protects existing classes. Last 15 epochs have augmentation disabled for confidence calibration.

**Auto-resume**: Writes a bash script that loops training with `--resume` if YOLOX crashes during no-aug epochs.

**Expected time: ~1-2 hours on T4**

In [ ]:
import shutil

PHASE2_OUTPUT_DIR = '/content/outputs/yolox_msl_finetune_phase2'
PHASE1_CKPT = f'{PHASE1_OUTPUT_DIR}/latest_ckpt.pth'

print("=" * 60)
print(f"PHASE 2: FULL FINE-TUNE ({PHASE2_EPOCHS} EPOCHS)")
print(f"  Backbone: UNFROZEN (all layers trainable)")
print(f"  LR: {PHASE2_LR * 64:.6f}")
print(f"  No-aug: last {NO_AUG_EPOCHS} epochs")
print("=" * 60)

# Write a bash script that handles auto-resume
train_script = f'''#!/bin/bash
cd /content/YOLOX
export PYTHONPATH=/content/YOLOX

# First run: -c loads Phase 1 weights, starts fresh from epoch 0
echo "=== First run: loading Phase 1 weights ==="
python tools/train.py \
    -f exps/msl_phase2_exp.py \
    -d 1 \
    -b {BATCH_SIZE} \
    --fp16 \
    -o \
    -c {PHASE1_CKPT}

# If it crashed, auto-resume until done
MAX_RETRIES=30
attempt=1
while [ $? -ne 0 ] && [ $attempt -le $MAX_RETRIES ]; do
    echo ""
    echo "=== Crashed, auto-resume attempt $attempt ==="
    attempt=$((attempt + 1))
    python tools/train.py \
        -f exps/msl_phase2_exp.py \
        -d 1 \
        -b {BATCH_SIZE} \
        --fp16 \
        -o \
        --resume
done

echo ""
echo "=== Training script finished ==="
'''

with open('/content/phase2_train.sh', 'w') as f:
    f.write(train_script)

# Run with ! so Colab properly blocks and shows output
!bash /content/phase2_train.sh

print("\n" + "=" * 60)
print("PHASE 2 COMPLETE!")
print("=" * 60)

## Cell 10: Save Checkpoints to Drive

In [ ]:
import shutil

print("Saving checkpoints to Google Drive...\n")

# Save epoch checkpoints from Phase 2
for epoch in range(SAVE_INTERVAL, PHASE2_EPOCHS + 1, SAVE_INTERVAL):
    src = f'{PHASE2_OUTPUT_DIR}/epoch_{epoch}_ckpt.pth'
    dst = f'{DRIVE_OUTPUT}/msl_finetune_epoch_{epoch}.pth'
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(src) / 1024 / 1024
        print(f"  msl_finetune_epoch_{epoch}.pth ({size_mb:.1f} MB)")

# Save latest
latest_src = f'{PHASE2_OUTPUT_DIR}/latest_ckpt.pth'
if os.path.exists(latest_src):
    shutil.copy(latest_src, f'{DRIVE_OUTPUT}/msl_finetune_final.pth')
    print(f"  msl_finetune_final.pth")

# Save class mapping
shutil.copy(f'{LOCAL_DATA_DIR}/class_mapping.json', f'{DRIVE_OUTPUT}/class_mapping_98class.json')
print(f"  class_mapping_98class.json")

print(f"\nAll checkpoints saved to: {DRIVE_OUTPUT}")
print("\nFiles on Drive:")
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    fpath = os.path.join(DRIVE_OUTPUT, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {f} ({size:.1f} MB)")

## Cell 11: MSL Validation + Regression Check

Tests the fine-tuned model on:
1. **MSL validation images** — does it detect military shipping labels?
2. **Original validation images** — are existing classes still working?

In [ ]:
import json, cv2, numpy as np, torch
from pathlib import Path
import matplotlib.pyplot as plt
from yolox.exp import get_exp
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

# Load class names
with open(f'{LOCAL_DATA_DIR}/class_mapping.json', 'r') as f:
    class_map = json.load(f)
if isinstance(class_map, list):
    CLASSES = [c['name'] for c in class_map]
elif isinstance(class_map, dict):
    CLASSES = [class_map[str(i)]['name'] for i in range(len(class_map))]

COLORS = np.random.randint(0, 255, size=(len(CLASSES), 3), dtype=np.uint8)
# Make MSL color bright red for visibility
COLORS[MSL_CLASS_ID] = [255, 0, 0]
preproc = ValTransform(legacy=False)

# Load final model
final_ckpt = f'{DRIVE_OUTPUT}/msl_finetune_final.pth'
if not os.path.exists(final_ckpt):
    final_ckpt = f'{PHASE2_OUTPUT_DIR}/latest_ckpt.pth'

exp = get_exp('/content/YOLOX/exps/msl_phase2_exp.py', None)
model = exp.get_model()
ckpt = torch.load(final_ckpt, map_location='cuda')
model.load_state_dict(ckpt['model'])
model.cuda().eval()
print(f"Model loaded from: {final_ckpt}")

def run_inference(img_path, conf_thresh=0.25):
    """Run inference on a single image, return detections."""
    img = cv2.imread(str(img_path))
    if img is None:
        return None, [], [], []
    h, w = img.shape[:2]
    tensor, _ = preproc(img, None, (640, 640))
    tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()
    with torch.no_grad():
        out = model(tensor)
        out = postprocess(out, NEW_NUM_CLASSES, conf_thresh, 0.45)
    if out[0] is not None:
        det = out[0].cpu().numpy()
        scale = min(640/h, 640/w)
        boxes = det[:, :4] / scale
        scores = det[:, 4] * det[:, 5]
        cls_ids = det[:, 6].astype(int)
        return img, boxes, scores, cls_ids
    return img, [], [], []

# ============================================
# TEST 1: MSL Validation Images
# ============================================
print("\n" + "=" * 60)
print("TEST 1: MSL DETECTION")
print("=" * 60)

val_dir = Path(f'{LOCAL_DATA_DIR}/images/val')
msl_val_images = sorted(val_dir.glob('msl_*'))

msl_detected = 0
msl_total_images = len(msl_val_images)
msl_confidences = []

for img_path in msl_val_images:
    img, boxes, scores, cls_ids = run_inference(img_path, conf_thresh=0.10)
    if len(cls_ids) > 0:
        msl_mask = cls_ids == MSL_CLASS_ID
        if msl_mask.any():
            msl_detected += 1
            msl_confidences.extend(scores[msl_mask].tolist())
            best_conf = scores[msl_mask].max()
            print(f"  {img_path.name}: MSL detected (conf={best_conf:.2f})")
        else:
            print(f"  {img_path.name}: No MSL (detected {len(cls_ids)} other objects)")
    else:
        print(f"  {img_path.name}: No detections")

print(f"\nMSL Detection Rate: {msl_detected}/{msl_total_images}")
if msl_confidences:
    print(f"MSL Confidence: min={min(msl_confidences):.2f}, max={max(msl_confidences):.2f}, mean={np.mean(msl_confidences):.2f}")

# ============================================
# TEST 2: Regression Check (original images)
# ============================================
print("\n" + "=" * 60)
print("TEST 2: REGRESSION CHECK (existing classes)")
print("=" * 60)

orig_val_images = sorted([p for p in val_dir.iterdir() if not p.name.startswith('msl_')])[:20]

all_confidences = []
false_msl = 0

for img_path in orig_val_images:
    img, boxes, scores, cls_ids = run_inference(img_path, conf_thresh=0.25)
    if len(cls_ids) > 0:
        all_confidences.extend(scores.tolist())
        msl_mask = cls_ids == MSL_CLASS_ID
        if msl_mask.any():
            false_msl += 1
            print(f"  WARNING: False MSL in {img_path.name} (conf={scores[msl_mask].max():.2f})")

print(f"\nOriginal images tested: {len(orig_val_images)}")
print(f"False MSL detections: {false_msl}")
if all_confidences:
    print(f"Existing class confidence: min={min(all_confidences):.2f}, max={max(all_confidences):.2f}, mean={np.mean(all_confidences):.2f}")

# ============================================
# SUMMARY
# ============================================
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
if msl_confidences:
    print(f"  MSL detection rate: {msl_detected}/{msl_total_images} ({msl_detected/max(1,msl_total_images)*100:.0f}%)")
    print(f"  MSL avg confidence: {np.mean(msl_confidences)*100:.1f}%")
print(f"  False MSL on orig images: {false_msl}/{len(orig_val_images)}")
if all_confidences:
    print(f"  Existing class avg conf: {np.mean(all_confidences)*100:.1f}%")
print("=" * 60)

## Cell 12: Visual Comparison Across Checkpoints

Compare MSL detection at different training stages.

In [ ]:
COMPARE_EPOCHS = [5, 10, 15, 20, 25]
CONF_THRESH = 0.10

# Get MSL validation images for comparison
compare_images = sorted(val_dir.glob('msl_*'))[:4]
if len(compare_images) < 4:
    compare_images.extend(sorted([p for p in val_dir.iterdir() if not p.name.startswith('msl_')])[:4-len(compare_images)])

fig, axes = plt.subplots(len(COMPARE_EPOCHS), len(compare_images), figsize=(4*len(compare_images), 4*len(COMPARE_EPOCHS)))
if len(COMPARE_EPOCHS) == 1:
    axes = [axes]

for row, epoch in enumerate(COMPARE_EPOCHS):
    ckpt_path = f'{PHASE2_OUTPUT_DIR}/epoch_{epoch}_ckpt.pth'
    if not os.path.exists(ckpt_path):
        ckpt_path = f'{PHASE2_OUTPUT_DIR}/latest_ckpt.pth'
        if not os.path.exists(ckpt_path):
            print(f"Epoch {epoch}: no checkpoint found, skipping")
            continue

    # Load checkpoint
    exp_cfg = get_exp('/content/YOLOX/exps/msl_phase2_exp.py', None)
    m = exp_cfg.get_model()
    c = torch.load(ckpt_path, map_location='cuda')
    m.load_state_dict(c['model'])
    m.cuda().eval()

    for col, img_path in enumerate(compare_images):
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        tensor, _ = preproc(img, None, (640, 640))
        tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()

        with torch.no_grad():
            out = m(tensor)
            out = postprocess(out, NEW_NUM_CLASSES, CONF_THRESH, 0.45)

        num_det = 0
        msl_det = 0
        if out[0] is not None:
            det = out[0].cpu().numpy()
            scale = min(640/h, 640/w)
            for box, score, cls_id in zip(det[:,:4]/scale, det[:,4]*det[:,5], det[:,6].astype(int)):
                x0,y0,x1,y1 = map(int, box)
                color = tuple(map(int, COLORS[min(cls_id, len(COLORS)-1)]))
                thickness = 3 if cls_id == MSL_CLASS_ID else 2
                cv2.rectangle(img, (x0,y0), (x1,y1), color, thickness)
                label = f"{CLASSES[cls_id][:12]}:{score:.2f}"
                cv2.putText(img, label, (x0,y0-5), cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
                if cls_id == MSL_CLASS_ID:
                    msl_det += 1
            num_det = len(det)

        ax = axes[row][col] if len(compare_images) > 1 else axes[row]
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        title = f"E{epoch}: {num_det}det"
        if msl_det > 0:
            title += f" ({msl_det}MSL)"
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    print(f"Epoch {epoch}: rendered")

plt.suptitle('MSL Fine-Tuning: Checkpoint Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_OUTPUT}/msl_checkpoint_comparison.png', dpi=150)
plt.show()
print(f"Saved to {DRIVE_OUTPUT}/msl_checkpoint_comparison.png")

## Cell 13: Summary & Next Steps

In [ ]:
print("=" * 70)
print("MSL FINE-TUNING COMPLETE")
print("=" * 70)

print(f"\nOutput: {DRIVE_OUTPUT}")
print("\nCheckpoints:")
if os.path.exists(DRIVE_OUTPUT):
    for f in sorted(os.listdir(DRIVE_OUTPUT)):
        if f.endswith('.pth'):
            size = os.path.getsize(os.path.join(DRIVE_OUTPUT, f)) / 1024 / 1024
            print(f"  {f} ({size:.1f} MB)")

print(f"""
{'=' * 70}
NEXT STEPS
{'=' * 70}

1. EXPORT TO EXECUTORCH:
   - Use 04_export_confidence_boost.ipynb as template
   - Update: CHECKPOINT_DIR = '{DRIVE_OUTPUT}'
   - Update: NUM_CLASSES = {NEW_NUM_CLASSES}
   - Export best epoch checkpoint(s) to .pte

2. TEST IN APP:
   - Copy .pte to assets/models/
   - Update class_mapping.json (add militaryShippingLabel as class {MSL_CLASS_ID})
   - Update NUM_CLASSES to {NEW_NUM_CLASSES} in executorchService.ts
   - Rebuild: npx expo run:ios / npx expo run:android

3. IF MSL DETECTION IS WEAK:
   - Try more epochs (increase PHASE2_EPOCHS to 40+)
   - Increase MSL_OVERSAMPLE to 8-10x
   - Collect more MSL training images
   - Try higher Phase 1 LR (0.002 / 64)

4. IF EXISTING CLASSES DEGRADED:
   - Use an earlier epoch checkpoint (e.g., epoch 10 or 15)
   - Lower Phase 2 LR further
   - Increase PHASE1_EPOCHS (longer frozen backbone phase)

{'=' * 70}
""")